# Crop Production Prediction - Testing & Prediction Notebook

This notebook uses the **trained unified GradientBoosting model** to predict crop Production.

### What this notebook covers:
1. **Load Model** - Load the trained pipeline and metadata
2. **Single Prediction** - Predict production for one State + Crop combination
3. **Future Forecast** - Predict production for a range of future years (auto-fills missing inputs from historical data)
4. **Batch Prediction** - Predict for multiple inputs at once

### Prerequisites:
- Run `train.py` first to generate the model files in `models/` directory
- Dataset `state_wise_crop_yild.csv` should be in the `dataset/` directory

---
## 0. Imports & Configuration

In [1]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import joblib

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("All imports loaded successfully.")

All imports loaded successfully.


In [2]:
# =============================================================================
# FILE PATHS - Update these if your directory structure is different
# =============================================================================

# Path to the trained model and metadata
MODEL_PATH = r"D:\researrch\agree.culture.Ai\code\model\state_crop_yeild\models\production_model.pkl"
META_PATH  = r"D:\researrch\agree.culture.Ai\code\model\state_crop_yeild\models\model_metadata.json"

# Path to the original dataset (used for auto-filling historical defaults in forecast mode)
DATASET_PATH = r"D:\researrch\agree.culture.Ai\code\dataset\state_wise_crop_yild.csv"

# Verify files exist
for label, path in [("Model", MODEL_PATH), ("Metadata", META_PATH), ("Dataset", DATASET_PATH)]:
    status = "FOUND" if os.path.exists(path) else "NOT FOUND"
    print(f"  [{status}] {label}: {path}")

  [FOUND] Model: D:\researrch\agree.culture.Ai\code\model\state_crop_yeild\models\production_model.pkl
  [FOUND] Metadata: D:\researrch\agree.culture.Ai\code\model\state_crop_yeild\models\model_metadata.json
  [FOUND] Dataset: D:\researrch\agree.culture.Ai\code\dataset\state_wise_crop_yild.csv


---
## 1. Load Trained Model & Metadata

In [3]:
# =============================================================================
# Load the trained sklearn Pipeline (preprocessor + GradientBoostingRegressor)
# =============================================================================

pipeline = joblib.load(MODEL_PATH)
print(f"[OK] Model loaded from: {MODEL_PATH}")

# Load metadata (contains model info, available states/crops, metrics)
with open(META_PATH, "r") as f:
    metadata = json.load(f)

# Display model summary
print(f"\n--- Model Summary ---")
print(f"  Model Type    : {metadata['model_type']}")
print(f"  Target        : {metadata['target']}")
print(f"  R2 Score      : {metadata['metrics']['r2']:.4f}")
print(f"  RMSE          : {metadata['metrics']['rmse']:.2f}")
print(f"  MAE           : {metadata['metrics']['mae']:.2f}")
print(f"  Trained on    : {metadata['training_data']['rows']} rows")
print(f"  States        : {len(metadata['training_data']['states'])}")
print(f"  Crops         : {len(metadata['training_data']['crops'])}")
print(f"  Year Range    : {metadata['training_data']['year_range'][0]} - {metadata['training_data']['year_range'][1]}")
print(f"  Trained At    : {metadata['trained_at']}")

[OK] Model loaded from: D:\researrch\agree.culture.Ai\code\model\state_crop_yeild\models\production_model.pkl

--- Model Summary ---
  Model Type    : GradientBoosting
  Target        : Production
  R2 Score      : 0.9953
  RMSE          : 18975284.39
  MAE           : 1119069.51
  Trained on    : 19689 rows
  States        : 30
  Crops         : 55
  Year Range    : 1997 - 2020
  Trained At    : 2026-07-01T16:29:58.685889


In [4]:
# =============================================================================
# Display all available States, Crops, and Seasons the model was trained on
# Use these EXACT names when making predictions
# =============================================================================

print("--- Available States ---")
for i, state in enumerate(metadata['training_data']['states'], 1):
    print(f"  {i:2d}. {state}")

print(f"\n--- Available Seasons ---")
for season in metadata['training_data']['seasons']:
    print(f"  - {season}")

print(f"\n--- Available Crops (first 20 of {len(metadata['training_data']['crops'])}) ---")
for i, crop in enumerate(metadata['training_data']['crops'][:20], 1):
    print(f"  {i:2d}. {crop}")
if len(metadata['training_data']['crops']) > 20:
    print(f"  ... and {len(metadata['training_data']['crops']) - 20} more")

--- Available States ---
   1. Andhra Pradesh
   2. Arunachal Pradesh
   3. Assam
   4. Bihar
   5. Chhattisgarh
   6. Delhi
   7. Goa
   8. Gujarat
   9. Haryana
  10. Himachal Pradesh
  11. Jammu and Kashmir
  12. Jharkhand
  13. Karnataka
  14. Kerala
  15. Madhya Pradesh
  16. Maharashtra
  17. Manipur
  18. Meghalaya
  19. Mizoram
  20. Nagaland
  21. Odisha
  22. Puducherry
  23. Punjab
  24. Sikkim
  25. Tamil Nadu
  26. Telangana
  27. Tripura
  28. Uttar Pradesh
  29. Uttarakhand
  30. West Bengal

--- Available Seasons ---
  - Autumn
  - Kharif
  - Rabi
  - Summer
  - Whole Year
  - Winter

--- Available Crops (first 20 of 55) ---
   1. Arecanut
   2. Arhar/Tur
   3. Bajra
   4. Banana
   5. Barley
   6. Black pepper
   7. Cardamom
   8. Cashewnut
   9. Castor seed
  10. Coconut
  11. Coriander
  12. Cotton(lint)
  13. Cowpea(Lobia)
  14. Dry chillies
  15. Garlic
  16. Ginger
  17. Gram
  18. Groundnut
  19. Guar seed
  20. Horse-gram
  ... and 35 more


---
## 2. Single Prediction

Predict production for **one specific input**. You must provide:
- `State`, `Crop`, `Season` - must match training data names exactly
- `Crop_Year` - the year to predict for
- `Area` - cultivated area in hectares
- `Annual_Rainfall` - annual rainfall in mm
- `Fertilizer` - fertilizer usage
- `Pesticide` - pesticide usage

In [5]:
# =============================================================================
# CHANGE THESE VALUES to test different predictions
# =============================================================================

state   = "West Bengal"     # State name (must match training data)
crop    = "Coconut"         # Crop name (must match training data)
season  = "Whole Year"      # Season: Kharif, Rabi, Summer, Autumn, Winter, Whole Year
year    = 2025              # Year to predict for
area    = 520.0             # Area in hectares
rainfall   = 1852.9         # Annual rainfall in mm
fertilizer = 766879.86      # Fertilizer usage
pesticide  = 2497.98        # Pesticide usage

print(f"Input configured: {state} / {crop} / {season} / {year}")

Input configured: West Bengal / Coconut / Whole Year / 2025


In [6]:
# =============================================================================
# Run single prediction
# =============================================================================

# Validate that inputs exist in training data
td = metadata["training_data"]
if state not in td["states"]:
    print(f"[WARN] State '{state}' not in training data!")
if crop not in td["crops"]:
    print(f"[WARN] Crop '{crop}' not in training data!")
if season not in td["seasons"]:
    print(f"[WARN] Season '{season}' not in training data!")

# Build the input DataFrame with all required features
input_df = pd.DataFrame([{
    "State":           state,
    "Crop":            crop,
    "Season":          season,
    "Crop_Year":       year,
    "Area":            area,
    "Annual_Rainfall": rainfall,
    "Fertilizer":      fertilizer,
    "Pesticide":       pesticide,
}])

print("Input data:")
display(input_df)

# Predict (clip to 0 since production can't be negative)
predicted_production = max(0, pipeline.predict(input_df)[0])

# Calculate estimated yield = Production / Area
estimated_yield = predicted_production / area if area > 0 else 0

print(f"\n{'='*50}")
print(f"  PREDICTION RESULTS")
print(f"{'='*50}")
print(f"  State              : {state}")
print(f"  Crop               : {crop}")
print(f"  Season             : {season}")
print(f"  Year               : {year}")
print(f"  Predicted Production: {predicted_production:,.2f}")
print(f"  Estimated Yield     : {estimated_yield:.4f}")
print(f"{'='*50}")

Input data:


,State,Crop,Season,Crop_Year,Area,Annual_Rainfall,Fertilizer,Pesticide
0,West Bengal,Coconut,Whole Year,2025,520.0,1852.9,766879.86,2497.98



  PREDICTION RESULTS
  State              : West Bengal
  Crop               : Coconut
  Season             : Whole Year
  Year               : 2025
  Predicted Production: 362,834,547.47
  Estimated Yield     : 697758.7451


---
## 3. Future Forecast

Predict production for a **range of future years** for a given State + Crop.

**How it works:**
- You provide just `State`, `Crop`, `start_year`, and `end_year`
- The script automatically fills in `Area`, `Rainfall`, `Fertilizer`, `Pesticide` from the **last 3 years of historical data** in the dataset
- You can override any of these auto-filled values

> **Note:** Tree-based models (GBR/RF) may give constant predictions across years when all other inputs are held constant, since they don't extrapolate time trends. This is a known limitation.

In [7]:
# =============================================================================
# FORECAST CONFIGURATION - Change these values
# =============================================================================

forecast_state = "Assam"          # State to forecast for
forecast_crop  = "Arecanut"       # Crop to forecast for
start_year     = 2025             # Start year of forecast
end_year       = 2030             # End year of forecast

# Set to None to auto-fill from historical data, or provide your own values
forecast_season     = None        # None = auto-detect from data
forecast_area       = None        # None = use historical average
forecast_rainfall   = None        # None = use historical average
forecast_fertilizer = None        # None = use historical average
forecast_pesticide  = None        # None = use historical average

print(f"Forecast configured: {forecast_state} / {forecast_crop} / {start_year}-{end_year}")

Forecast configured: Assam / Arecanut / 2025-2030


In [8]:
# =============================================================================
# Run forecast - loads historical data and predicts future production
# =============================================================================

# Step 1: Load and clean the dataset to get historical averages
hist_df = pd.read_csv(DATASET_PATH)
for col in hist_df.select_dtypes(include=["object", "string"]).columns:
    hist_df[col] = hist_df[col].str.strip()

# Step 2: Filter for the target State + Crop
mask = (hist_df["State"] == forecast_state) & (hist_df["Crop"] == forecast_crop)
if forecast_season is not None:
    mask &= (hist_df["Season"] == forecast_season)

subset = hist_df[mask]

if subset.empty:
    print(f"[ERROR] No historical data found for State='{forecast_state}', Crop='{forecast_crop}'.")
    print(f"Check available states/crops in cell above.")
else:
    # Step 3: Compute defaults from last 3 years of available data
    recent_years = sorted(subset["Crop_Year"].unique())[-3:]
    recent = subset[subset["Crop_Year"].isin(recent_years)]

    # Use provided values or fall back to historical averages
    fc_area       = forecast_area       if forecast_area       is not None else float(recent["Area"].mean())
    fc_rainfall   = forecast_rainfall   if forecast_rainfall   is not None else float(recent["Annual_Rainfall"].mean())
    fc_fertilizer = forecast_fertilizer if forecast_fertilizer is not None else float(recent["Fertilizer"].mean())
    fc_pesticide  = forecast_pesticide  if forecast_pesticide  is not None else float(recent["Pesticide"].mean())
    fc_season     = forecast_season     if forecast_season     is not None else recent["Season"].mode().iloc[0]

    print(f"Historical defaults (based on years {list(recent_years)}):")
    print(f"  Season    : {fc_season}")
    print(f"  Area      : {fc_area:.2f}")
    print(f"  Rainfall  : {fc_rainfall:.2f}")
    print(f"  Fertilizer: {fc_fertilizer:.2f}")
    print(f"  Pesticide : {fc_pesticide:.2f}")

    # Step 4: Build input DataFrame for all forecast years
    years = list(range(start_year, end_year + 1))
    forecast_rows = []
    for yr in years:
        forecast_rows.append({
            "State":           forecast_state,
            "Crop":            forecast_crop,
            "Season":          fc_season,
            "Crop_Year":       yr,
            "Area":            fc_area,
            "Annual_Rainfall": fc_rainfall,
            "Fertilizer":      fc_fertilizer,
            "Pesticide":       fc_pesticide,
        })

    forecast_input_df = pd.DataFrame(forecast_rows)

    # Step 5: Predict (clip negatives to 0)
    predictions = np.maximum(0, pipeline.predict(forecast_input_df))

    # Step 6: Build results table
    forecast_results = pd.DataFrame({
        "Year": years,
        "Predicted_Production": np.round(predictions, 2),
        "Estimated_Yield": np.round(predictions / fc_area, 4) if fc_area > 0 else 0,
    })

    print(f"\n{'='*55}")
    print(f"  FORECAST: {forecast_state} / {forecast_crop} ({fc_season})")
    print(f"{'='*55}")
    display(forecast_results)
    print(f"{'='*55}")

Historical defaults (based on years [np.int64(2017), np.int64(2018), np.int64(2019)]):
  Season    : Rabi
  Area      : 67165.00
  Rainfall  : 2185.39
  Fertilizer: 11001975.25
  Pesticide : 24626.52

  FORECAST: Assam / Arecanut (Rabi)


,Year,Predicted_Production,Estimated_Yield
0,2025,83416.22,1.242
1,2026,83416.22,1.242
2,2027,83416.22,1.242
3,2028,83416.22,1.242
4,2029,83416.22,1.242
5,2030,83416.22,1.242


---
## 4. Batch Prediction - Multiple Crops at Once

Test the model with **multiple input rows** simultaneously.
Each row needs: `State`, `Crop`, `Season`, `Crop_Year`, `Area`, `Annual_Rainfall`, `Fertilizer`, `Pesticide`

In [9]:
# =============================================================================
# Define multiple test inputs - add/modify rows as needed
# =============================================================================

batch_inputs = [
    # West Bengal crops
    {"State": "West Bengal", "Crop": "Rice",       "Season": "Kharif",     "Crop_Year": 2025, "Area": 5500000, "Annual_Rainfall": 1852.9, "Fertilizer": 766879.86, "Pesticide": 2497.98},
    {"State": "West Bengal", "Crop": "Potato",     "Season": "Rabi",       "Crop_Year": 2025, "Area":  400000, "Annual_Rainfall": 1852.9, "Fertilizer": 766879.86, "Pesticide": 2497.98},
    {"State": "West Bengal", "Crop": "Jute",       "Season": "Kharif",     "Crop_Year": 2025, "Area":  600000, "Annual_Rainfall": 1852.9, "Fertilizer": 766879.86, "Pesticide": 2497.98},
    # Assam crops
    {"State": "Assam",       "Crop": "Arecanut",   "Season": "Whole Year", "Crop_Year": 2025, "Area":   67165, "Annual_Rainfall": 2185.4, "Fertilizer": 11001975.25, "Pesticide": 24626.52},
    {"State": "Assam",       "Crop": "Rice",       "Season": "Kharif",     "Crop_Year": 2025, "Area": 2500000, "Annual_Rainfall": 2185.4, "Fertilizer": 11001975.25, "Pesticide": 24626.52},
    # Punjab crops
    {"State": "Punjab",      "Crop": "Wheat",      "Season": "Rabi",       "Crop_Year": 2025, "Area": 3500000, "Annual_Rainfall":  649.0, "Fertilizer": 5678900.00,  "Pesticide": 18500.00},
]

batch_df = pd.DataFrame(batch_inputs)
print(f"Batch input: {len(batch_df)} rows")
display(batch_df)

Batch input: 6 rows


,State,Crop,Season,Crop_Year,Area,Annual_Rainfall,Fertilizer,Pesticide
0,West Bengal,Rice,Kharif,2025,5500000,1852.9,766879.86,2497.98
1,West Bengal,Potato,Rabi,2025,400000,1852.9,766879.86,2497.98
2,West Bengal,Jute,Kharif,2025,600000,1852.9,766879.86,2497.98
3,Assam,Arecanut,Whole Year,2025,67165,2185.4,11001975.25,24626.52
4,Assam,Rice,Kharif,2025,2500000,2185.4,11001975.25,24626.52
5,Punjab,Wheat,Rabi,2025,3500000,649.0,5678900.00,18500.00


In [10]:
# =============================================================================
# Run batch prediction
# =============================================================================

# Required feature columns (must match training pipeline)
required_cols = ["State", "Crop", "Season", "Crop_Year", "Area",
                 "Annual_Rainfall", "Fertilizer", "Pesticide"]

# Predict production for all rows (clip negatives to 0)
batch_predictions = np.maximum(0, pipeline.predict(batch_df[required_cols]))

# Add predictions to the DataFrame
results_df = batch_df.copy()
results_df["Predicted_Production"] = np.round(batch_predictions, 2)
results_df["Estimated_Yield"] = np.where(
    results_df["Area"] > 0,
    np.round(results_df["Predicted_Production"] / results_df["Area"], 4),
    0
)

print(f"\n{'='*60}")
print(f"  BATCH PREDICTION RESULTS ({len(results_df)} rows)")
print(f"{'='*60}")

# Display a clean summary table
summary = results_df[["State", "Crop", "Season", "Crop_Year",
                       "Predicted_Production", "Estimated_Yield"]].copy()
display(summary)


  BATCH PREDICTION RESULTS (6 rows)


,State,Crop,Season,Crop_Year,Predicted_Production,Estimated_Yield
0,West Bengal,Rice,Kharif,2025,10591237.32,1.9257
1,West Bengal,Potato,Rabi,2025,3266033.25,8.1651
2,West Bengal,Jute,Kharif,2025,2627814.38,4.3797
3,Assam,Arecanut,Whole Year,2025,109410.83,1.6290
4,Assam,Rice,Kharif,2025,6250196.85,2.5001
5,Punjab,Wheat,Rabi,2025,13356725.79,3.8162


---
## 5. Validate Against Historical Data

Compare model predictions against **known actual values** from the dataset to see how accurate the model is.

In [11]:
# =============================================================================
# Load dataset and pick some random rows to compare predictions vs actuals
# =============================================================================

val_df = pd.read_csv(DATASET_PATH)
for col in val_df.select_dtypes(include=["object", "string"]).columns:
    val_df[col] = val_df[col].str.strip()

# Sample 15 random rows for validation
sample_rows = val_df.sample(n=15, random_state=123)

# Get the features the model expects
feature_cols = ["State", "Crop", "Season", "Crop_Year", "Area",
                "Annual_Rainfall", "Fertilizer", "Pesticide"]

# Predict on these known rows
val_predictions = np.maximum(0, pipeline.predict(sample_rows[feature_cols]))

# Build comparison table
comparison = pd.DataFrame({
    "State":              sample_rows["State"].values,
    "Crop":               sample_rows["Crop"].values,
    "Year":               sample_rows["Crop_Year"].values,
    "Actual_Production":  sample_rows["Production"].values,
    "Predicted":          np.round(val_predictions, 2),
})
comparison["Error_%"] = np.where(
    comparison["Actual_Production"] != 0,
    np.round(abs(comparison["Predicted"] - comparison["Actual_Production"]) / abs(comparison["Actual_Production"]) * 100, 2),
    np.nan
)

print("Model Validation: Predicted vs Actual Production\n")
display(comparison)

# Summary stats
avg_error = comparison["Error_%"].mean()
median_error = comparison["Error_%"].median()
print(f"\nAverage Error: {avg_error:.2f}%")
print(f"Median Error:  {median_error:.2f}%")

Model Validation: Predicted vs Actual Production



,State,Crop,Year,Actual_Production,Predicted,Error_%
0,Tamil Nadu,Other Cereals,2016,210,0.00,100.00
1,Nagaland,Sesamum,2006,3460,0.00,100.00
2,Mizoram,Maize,2019,765,31172.67,3974.86
3,West Bengal,Mesta,2007,77499,76485.62,1.31
4,Odisha,Moong(Green Gram),2018,17222,29800.15,73.04
5,Jammu and Kashmir,Maize,2009,486988,617180.27,26.73
6,Gujarat,Rice,2007,1376300,1272682.75,7.53
7,Chhattisgarh,Cotton(lint),2009,0,19904.28,NaN
8,Manipur,Turmeric,2003,380,2815.01,640.79
9,Puducherry,Small millets,2013,3,0.00,100.00



Average Error: 370.59%
Median Error:  46.16%


---
## 6. Quick Custom Test

Use this cell for quick ad-hoc tests. Just change the values and re-run.

In [12]:
# =============================================================================
# Quick custom prediction - change values and run this cell
# =============================================================================

quick_input = pd.DataFrame([{
    "State":           "Karnataka",
    "Crop":            "Rice",
    "Season":          "Kharif",
    "Crop_Year":       2026,
    "Area":            1500000,
    "Annual_Rainfall": 1248.0,
    "Fertilizer":      4500000.0,
    "Pesticide":       12000.0,
}])

result = max(0, pipeline.predict(quick_input)[0])
print(f"State: {quick_input['State'].values[0]}")
print(f"Crop:  {quick_input['Crop'].values[0]}")
print(f"Year:  {quick_input['Crop_Year'].values[0]}")
print(f"\n=> Predicted Production: {result:,.2f}")
if quick_input['Area'].values[0] > 0:
    print(f"=> Estimated Yield:      {result / quick_input['Area'].values[0]:.4f}")

State: Karnataka
Crop:  Rice
Year:  2026

=> Predicted Production: 1,240,390.32
=> Estimated Yield:      0.8269
